In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import torch

import warnings
warnings.filterwarnings("ignore")

In [ ]:
#----------------------------- DON'T CHANGE THIS --------------------------
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB=20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

In [ ]:
# CONFIGURATION
DATA_ROOT = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock'] # Make the list of all genres available (alphabetical order)
STEMS = {'drums': 'drums.wav', 'vocals': 'vocals.wav', 'bass': 'bass.wav', 'other': 'other.wav'} # Write here stems file name
STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
GENRE_TO_TEST = 'rock'
# SONG_INDEX = #Enter index as per Q10.

In [ ]:
import os
import random
from pathlib import Path

# --- CONFIGURATION ---
DATA_ROOT = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']
STEMS = {'drums': 'drums.wav', 'vocals': 'vocals.wav', 'bass': 'bass.wav', 'other': 'other.wav'}
STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
GENRE_TO_TEST = 'rock'

def build_dataset(root_dir, val_split=0.17, seed=42):
    # Initialize the nested dictionary structure
    train_dataset = {g: {k: [] for k in STEM_KEYS} for g in GENRES}
    val_dataset   = {g: {k: [] for k in STEM_KEYS} for g in GENRES}

    rng = random.Random(seed)
    CORRUPT_LIMIT = 4096 # 4KB
    
    # Path to the specific subfolder shown in your image
    base_path = os.path.join(root_dir, 'genres_stems')

    for genre in GENRES:
        genre_path = os.path.join(base_path, genre)

        if not os.path.isdir(genre_path):
            continue

        valid_songs = []

        # 1. Gather all valid song folders (e.g., blues.00000)
        for song_folder in sorted(os.listdir(genre_path)):
            song_path = os.path.join(genre_path, song_folder)

            if not os.path.isdir(song_path):
                continue

            # Check if all required stems exist and are not corrupted
            song_stems = {}
            is_valid = True

            for key, filename in STEMS.items():
                stem_path = os.path.join(song_path, filename)
                
                if not os.path.isfile(stem_path) or os.path.getsize(stem_path) < CORRUPT_LIMIT:
                    is_valid = False
                    break
                
                song_stems[key] = stem_path

            if is_valid:
                valid_songs.append(song_stems)

        # 2. Shuffle the full song groups (keeping stems together)
        rng.shuffle(valid_songs)

        # 3. Calculate Split
        split_idx = int(len(valid_songs) * (1 - val_split))
        train_songs = valid_songs[:split_idx]
        val_songs = valid_songs[split_idx:]
        
        # This confirms why you see '6' (it prints per genre)
        print(f"Genre: {genre:10} | Train: {len(train_songs):3} | Val: {len(val_songs):3}")

         # 4. Helper to populate the nested dictionary structure
        def add_to_dict(target_dict, song_list):
            for song_entry in song_list:
                for key in STEM_KEYS:
                    # Append the path to the correct genre and stem key
                    target_dict[genre][key].append(song_entry[key])

        add_to_dict(train_dataset, train_songs)
        add_to_dict(val_dataset, val_songs)

    return train_dataset, val_dataset

# Execute
tr, val = build_dataset(DATA_ROOT)


In [ ]:
# What is the mean duration (in seconds) of the Jazz genre stems in train (genres_stems) dataset?

import soundfile as sf
import numpy as np

genre_name = 'jazz' 
durations = []

jazz_data = tr[genre_name]

for stem_key, file_paths in jazz_data.items():
    for path in file_paths:
        try:
            audio_info = sf.info(path)
            durations.append(audio_info.duration)
        except Exception as e:
            print(f"Error")

if durations:
    mean_duration = np.mean(durations)
    print(f"Mean Duration: {mean_duration:.2f} seconds")
else:
    print(f"No audio files could be processed")

In [ ]:
# What are the unique sample rates present in the entire dataset? (consider genre stems, noise data and mashups) Enter your answer as a comma separated list. Example: [40000, 50000, 60000]

import soundfile as sf

def get_unique_sample_rates(dataset):
    unique_srs = set()
    
    for genre, stems in dataset.items():
        for stem_key, file_paths in stems.items():
            for path in file_paths:
                try:
                    sr = sf.info(path).samplerate
                    unique_srs.add(sr)
                except Exception as e:
                    print(f"Skipping")
                    
    return sorted(list(unique_srs))

sample_rates = get_unique_sample_rates(tr)

print(sample_rates)

In [ ]:
# What is the average peak amplitude (in dB) for vocal stems in train dataset? 

import soundfile as sf
import numpy as np
import warnings

def calculate_average_vocal_peak_db(dataset):
    peak_dbs = []
        
    for genre in dataset.keys():
        if 'vocals' in dataset[genre]:
            vocal_paths = dataset[genre]['vocals']
            
            for path in vocal_paths:
                try:
                    data, samplerate = sf.read(path)
                    
                    peak_amplitude = np.max(np.abs(data))
                    
                    if peak_amplitude > 0:
                        peak_db = 20 * np.log10(peak_amplitude)
                    else:
                        peak_db = -100.0 
                        
                    peak_dbs.append(peak_db)
                    
                except Exception as e:
                    print(f"Error")
    
    if peak_dbs:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            average_peak_db = np.mean(peak_dbs)
            
        print(f"Average Peak Amplitude: {average_peak_db:.2f}")
        return average_peak_db
    else:
        print("Error")
        return None

avg_vocal_peak_db = calculate_average_vocal_peak_db(tr)

In [ ]:
# What is the mean spectral centroid for 'blues' genre in the train dataset?  

import librosa
import numpy as np
import warnings

def calculate_mean_spectral_centroid(dataset, genre_name):
    genre_name = genre_name.lower()
    if genre_name not in dataset:
        print(f"Genre not found")
        return None

    track_centroids = []

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        
        for stem_key, file_paths in dataset[genre_name].items():
            for path in file_paths:
                try:
                    y, sr = librosa.load(path, sr=None)
                    
                    if np.max(np.abs(y)) == 0:
                        continue
                        
                    cent = librosa.feature.spectral_centroid(y=y, sr=sr)
                    
                    track_mean = np.mean(cent)
                    track_centroids.append(track_mean)
                    
                except Exception as e:
                    print(f"Error")

    if track_centroids:
        grand_mean_centroid = np.mean(track_centroids)
        print(f"Mean Spectral Centroid for '{genre_name}': {grand_mean_centroid:.2f}")
        return grand_mean_centroid
    else:
        print(f"No valid audio files found")
        return None

mean_blues_centroid = calculate_mean_spectral_centroid(tr, 'blues')

In [ ]:
# Which genre in the train dataset has the highest mean spectral centroid?

import librosa
import numpy as np
import warnings

def find_highest_spectral_centroid_genre(dataset):
    genre_centroids = {}
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        
        for genre, stems in dataset.items():
            track_centroids = []
            
            for stem_key, file_paths in stems.items():
                for path in file_paths:
                    try:
                        y, sr = librosa.load(path, sr=None)
                        
                        if np.max(np.abs(y)) == 0:
                            continue
                            
                        cent = librosa.feature.spectral_centroid(y=y, sr=sr)
                        track_centroids.append(np.mean(cent))
                        
                    except Exception as e:
                        pass 
            
            if track_centroids:
                mean_centroid = np.mean(track_centroids)
                genre_centroids[genre] = mean_centroid
                
    if genre_centroids:
        highest_genre = max(genre_centroids, key=genre_centroids.get)
        highest_value = genre_centroids[highest_genre]
        
        print(f"Genre: '{highest_genre}'")
        print(f"Value: {highest_value:.2f}")
        
        return highest_genre
    else:
        print("No valid audio files.")
        return None

highest_genre = find_highest_spectral_centroid_genre(tr)

In [ ]:
# How many stem audio files in the train dataset contain silence in the first 0.5 seconds?  

import soundfile as sf
import numpy as np

def count_initial_silence_threshold(dataset, duration_sec=0.5, threshold=1e-4):
  
    silent_count = 0
    total_files = 0
    
    
    for genre, stems in dataset.items():
        for stem_key, file_paths in stems.items():
            for path in file_paths:
                total_files += 1
                try:
                    info = sf.info(path)
                    frames_to_read = int(info.samplerate * duration_sec)
                    
                    data, sr = sf.read(path, frames=frames_to_read, dtype='float32')
                    
                    if len(data) > 0 and np.max(np.abs(data)) < threshold:
                        silent_count += 1
                        
                except Exception as e:
                    print(f"Skipping {path} due to error: {e}")
                    
    print(f"Files with silence: {silent_count}")
    
    return silent_count

silent_files_count = count_initial_silence_threshold(tr, duration_sec=0.5, threshold=1e-4)

In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, confusion_matrix, classification_report

# --- 1. Setup and Preprocessing ---
ROOT = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
STEMS_PATH = os.path.join(ROOT, 'genres_stems')
GENRES = ["blues", "classical", "country", "disco", "hiphop", "jazz", "metal", "pop", "reggae", "rock"]

def extract_features(song_path):
    # Load 10s at 22050Hz
    y, sr = librosa.load(os.path.join(song_path, 'other.wav'), sr=22050, duration=10)
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    spec_cent = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    zcr = np.mean(librosa.feature.zero_crossing_rate(y))
    rolloff = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
    return [float(tempo), spec_cent, zcr, rolloff]

# --- 2. Data Preparation & Stratified Split ---
data = []
for g in GENRES:
    gp = os.path.join(STEMS_PATH, g)
    songs = [s for s in os.listdir(gp) if os.path.isdir(os.path.join(gp, s))]
    for s in songs[:50]: # Sampling 50 for speed; use all for final
        data.append({'path': os.path.join(gp, s), 'genre': g})

df = pd.DataFrame(data)
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['genre'], random_state=42)

# --- 3. Model Training (Decision Tree) ---
X_train = np.array([extract_features(p) for p in train_df['path']])
y_train = train_df['genre']
X_val = np.array([extract_features(p) for p in val_df['path']])
y_val = val_df['genre']

clf = DecisionTreeClassifier(max_depth=5, random_state=42)
clf.fit(X_train, y_train)


# MY CODE 
# COMPUTE PREDICTED VALUES
y_pred = clf.predict(X_val)

# COMPUTE VALIDATION MACRO F1 SCORE
macro_f1 = f1_score(y_val, y_pred, average='macro')

# COMPUTE CONFUSION MATRIX
cm = confusion_matrix(y_val, y_pred, labels=GENRES)

# COMPUTE CLASSIFICATION REPORT
cr = classification_report(y_val, y_pred, labels=GENRES, target_names=GENRES)

print(f"Validation Macro F1 Score: {macro_f1:.4f}\n")
print("Detailed Classification Report:")
print(cr)

# Visualization of Confusion Matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=GENRES, yticklabels=GENRES)
plt.title('Confusion Matrix - Decision Tree (Other Stems)')
plt.xlabel('Predicted Genre')
plt.ylabel('True Genre')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()